In [2]:
!pip install -U yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.52
    Uninstalling yfinance-0.2.52:
      Successfully uninstalled yfinance-0.2.52


In [30]:
import yfinance as yf
import pandas as pd
import numpy as np


# Net Income (순이익) 추출 함수
def get_net_income_from_operatings(stock, q_data=True):
  if q_data:
    financials = stock.quarterly_financials.T
  else:
    financials = stock.financials.T
  return financials["Net Income Continuous Operations"]

# Operating Cash Flow (영업활동 현금흐름) 추출 함수
def get_operating_cashflow(stock, q_data=True):
  if q_data:
    cashflow = stock.quarterly_cashflow.T
  else:
    cashflow = stock.cashflow.T
  return cashflow["Cash Flow From Continuing Operating Activities"]

def get_total_asset(stock, q_data=True):
  if q_data:
    balancesheet = stock.quarterly_balancesheet.T
  else:
    balancesheet = stock.balancesheet.T
  return balancesheet['Total Assets']

def get_net_income(stock, q_data=True):
  if q_data:
    financials = stock.quarterly_financials.T
  else:
    financials = stock.financials.T
  return financials['Net Income']

# Cash Dividends Paid (배당금 지급) 추출 함수
def get_cash_dividends_paid(stock, q_data=True):
  if q_data:
    cashflow = stock.quarterly_cashflow.T
  else:
    cashflow = stock.cashflow.T
  return cashflow["Cash Dividends Paid"]

# Repurchase of Capital Stock (자사주 매입) 추출 함수
def get_repurchase_of_stock(stock, q_data=True):
  if q_data:
    cashflow = stock.quarterly_cashflow.T
  else:
    cashflow = stock.cashflow.T
  return cashflow["Repurchase Of Capital Stock"]


# 모든 데이터를 종합하는 함수
def get_financial_data(ticke, q_date=True):
    stock = yf.Ticker(ticker)

    # 개별 함수 호출하여 데이터 추출
    net_income = get_net_income(stock, q_date)
    cash_dividends_paid = get_cash_dividends_paid(stock, q_date)
    repurchase_of_stock = get_repurchase_of_stock(stock, q_date)
    net_income_from_operatings = get_net_income_from_operatings(stock, q_date)
    cashflow_of_operating = get_operating_cashflow(stock, q_date)
    total_asset = get_total_asset(stock, q_date)

    # 데이터프레임 생성
    df = pd.DataFrame({
        'Net Income': net_income,
        'Cash Dividends Paid': cash_dividends_paid,
        'Repurchase of Capital Stock': repurchase_of_stock,
        'Net Income from Operating':net_income_from_operatings,
        'Cashflow of Operating':cashflow_of_operating,
        'Shareholder Return': cash_dividends_paid + repurchase_of_stock,
        '영업 활동으로 발생하는 현금 흐름 (Income/Operating) %': (net_income_from_operatings/cashflow_of_operating)*100,
        '자산 대비 수익 창출 능력 (Income/Assets) %': (net_income_from_operatings/total_asset)*100
    })
    df.index = df.index.strftime('%y') + "Q" + df.index.to_period("Q").astype(str).str[-1]
    return df

In [33]:
# 예시 종목: 애플 (AAPL)
ticker = "NKE"
df = get_financial_data(ticker)

In [34]:
df['영업 활동으로 발생하는 현금 흐름 (Income/Operating) %'].sort_index()

,영업 활동으로 발생하는 현금 흐름 (Income/Operating) %
23Q2,NaN
23Q3,NaN
23Q4,56.017039
24Q1,56.920835
24Q2,57.273769
24Q3,266.751269
24Q4,110.867493


In [35]:
df['자산 대비 수익 창출 능력 (Income/Assets) %'].sort_index()

,자산 대비 수익 창출 능력 (Income/Assets) %
23Q2,NaN
23Q3,NaN
23Q4,4.241593
24Q1,3.137381
24Q2,3.935975
24Q3,2.775504
24Q4,3.063832


In [36]:
df/np.power(10,9)

,Net Income,Cash Dividends Paid,Repurchase of Capital Stock,Net Income from Operating,Cashflow of Operating,Shareholder Return,영업 활동으로 발생하는 현금 흐름 (Income/Operating) %,자산 대비 수익 창출 능력 (Income/Assets) %
24Q4,1.163,-0.557,-1.096,1.163,1.049,-1.653,0.0,0.0
24Q3,1.051,-0.558,-1.184,1.051,0.394,-1.742,0.0,0.0
24Q2,1.5,-0.56,-1.036,1.5,2.619,-1.596,0.0,0.0
24Q1,1.172,-0.562,-0.883,1.172,2.059,-1.445,0.0,0.0
23Q4,1.578,-0.523,-1.198,1.578,2.817,-1.721,0.0,0.0
23Q3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
